# Amaliy mashgʻulot 4. N-gram modellari va silliqlash

4-laboratoriyada bigrammalarni sanadingiz, k va d slayderlarini surdingiz, perpleksiyani hisobladingiz va ruletka bilan gaplar yaratdingiz. Bu yerda xuddi shuni kodda bajaramiz:

1. unigramma va bigrammalarni sanaymiz va maksimal haqiqatga oʻxshashlik bahosini (MLE) quramiz;
2. add-k (Laplas) va interpolyatsiyalangan Kneser–Ney silliqlashini noldan yozamiz va har bir taqsimot yigʻindisi 1 ga teng ekanini tekshiramiz;
3. test gaplarining perpleksiyasini hisoblaymiz va k hamda d ni tanlaymiz;
4. turli haroratda gaplar yaratamiz;
5. oʻz sonlarimizni `nltk.lm` bilan solishtiramiz.

*«NLP algoritmlari» kursining 4-laboratoriyasiga amaliy mashgʻulot. Kataklarni yuqoridan pastga ishga tushiring: Shift+Enter yoki «Runtime → Run all». Javoblaringizni saqlash uchun «File → Save a copy in Drive» ni bosing.*

## 1. Sanoqlar va MLE

Korpus laboratoriyadagi bilan bir xil: 9 ta qisqa gap, kichik harflarda va tinish belgilarisiz. Har bir gapni `<s>` va `</s>` xizmat tokenlari bilan oʻraymiz. Bigramma modeli quyidagini hisoblaydi:

$$P_{MLE}(w \mid v) = \frac{c(v\,w)}{c(v)},$$

bu yerda $c(v\,w)$ — juftlik necha marta uchragani, $c(v)$ esa $v$ necha marta kontekst boʻlgani. Bashorat lugʻati $V$ — barcha soʻzlar va `</s>`.

In [ ]:
corpus = [
    "mirzo ulugʻbek buyuk olim edi", "mirzo ulugʻbek samarqandda yashagan", "mirzo ulugʻbek buyuk hukmdor edi",
    "mirzo ulugʻbek rasadxona qurdirgan", "alisher navoiy buyuk shoir edi", "alisher navoiy samarqandda oʻqigan",
    "men samarqandda yashayman", "men choy ichaman", "onam choy ichadi",
]
CTX = "ulugʻbek"                 # laboratoriyadagi kontekst
GEN_CTX = "men"                  # 4-boʻlimdagi grafik uchun kontekst
PAIR = ("ulugʻbek", "samarqandda")   # A koʻp uchraydi, lekin faqat «mirzo»dan keyin; B turli soʻzlardan keyin
NEW_CTXS = ["onam", "alisher", "buyuk"]   # ulardan keyin «ulugʻbek» ham, «samarqandda» ham uchramagan

In [ ]:
import math
from collections import Counter
import numpy as np
import pandas as pd
pd.set_option("display.width", 200)

BOS, EOS = "<s>", "</s>"

class BigramCounts:
    def __init__(self, sentences):
        sents = [[BOS] + s.split() + [EOS] for s in sentences]
        self.words = []
        for s in sents:
            for w in s[1:-1]:
                if w not in self.words:
                    self.words.append(w)
        self.pred = self.words + [EOS]           # nima bashorat qilinadi
        self.hist = [BOS] + self.words           # nima kontekst boʻlishi mumkin
        self.V = len(self.pred)
        self.big = Counter((s[i - 1], s[i]) for s in sents for i in range(1, len(s)))
        self.uni = Counter(w for s in sents for w in s[1:])
        self.N = sum(self.uni.values())
        self.c = Counter()          # c(v)
        self.n1_after = Counter()   # N1+(v •)
        self.n1_before = Counter()  # N1+(• w)
        for (v, w), n in self.big.items():
            self.c[v] += n
            self.n1_after[v] += 1
            self.n1_before[w] += 1
        self.n1_all = len(self.big)  # N1+(• •)
        self.n_sents = len(sents)

def mle(m, w, v):
    return m.big[v, w] / m.c[v]

M = BigramCounts(corpus)
print(f"{M.n_sents} ta gap, {M.N} ta token (<s> siz), |V| (soʻzlar + </s>) = {M.V}, {M.n1_all} ta turli bigramma")
print(f"jadvalning toʻlgan kataklari: {M.n1_all} / {len(M.hist) * M.V} = {M.n1_all / (len(M.hist) * M.V):.1%}")
print("Eng koʻp uchraydigan soʻzlar:", M.uni.most_common(6))

Laboratoriyadagi kabi bigrammalar jadvali: satr — oldingi soʻz $v$, ustun — keyingi soʻz $w$. Kataklarning deyarli hammasi boʻsh, MLE uchun esa har bir boʻsh katak «imkonsiz» degani.

In [ ]:
import matplotlib.pyplot as plt

table = pd.DataFrame([[M.big[v, w] for w in M.pred] for v in M.hist], index=M.hist, columns=M.pred)
table["Σ"] = table.sum(axis=1)
print(table.to_string())

fig, ax = plt.subplots(figsize=(9, 6))
ax.imshow(table.drop(columns="Σ").values, cmap="Blues")
ax.set_xticks(range(M.V)); ax.set_xticklabels(M.pred, rotation=90)
ax.set_yticks(range(len(M.hist))); ax.set_yticklabels(M.hist)
for i, v in enumerate(M.hist):
    for j, w in enumerate(M.pred):
        if M.big[v, w]:
            ax.text(j, i, M.big[v, w], ha="center", va="center", fontsize=8,
                    color="white" if M.big[v, w] > 2 else "black")
ax.set_xlabel("w"); ax.set_ylabel("v"); ax.set_title("c(v w)")
plt.tight_layout()
plt.show()

## 2. Silliqlash: add-k va Kneser–Ney

**Add-k (k = 1 da Laplas)** har bir sanoqqa k qoʻshadi:
$$P_{add\text{-}k}(w \mid v) = \frac{c(v\,w) + k}{c(v) + k\,|V|}.$$

**Interpolyatsiyalangan Kneser–Ney** nolga teng boʻlmagan har bir sanoqdan $d$ chegirmani ayiradi va boʻshagan $\lambda(v)$ massani **davom ettirish** taqsimotiga beradi:
$$P_{KN}(w \mid v) = \frac{\max(c(v\,w) - d,\, 0)}{c(v)} + \lambda(v)\, P_{cont}(w), \quad \lambda(v) = \frac{d\cdot N_{1+}(v\,\bullet)}{c(v)}, \quad P_{cont}(w) = \frac{N_{1+}(\bullet\,w)}{N_{1+}(\bullet\,\bullet)}.$$

$N_{1+}(\bullet\,w)$ — $w$ dan oldin kelgan **turli** soʻzlar soni. Har bir kontekst uchun butun lugʻat boʻyicha ehtimollar yigʻindisi 1 ga teng ekanini tekshiramiz.

In [ ]:
def addk(m, w, v, k=1.0):
    return (m.big[v, w] + k) / (m.c[v] + k * m.V)

def p_cont(m, w):
    return m.n1_before[w] / m.n1_all

def lam(m, v, d=0.75):
    return d * m.n1_after[v] / m.c[v]

def kn(m, w, v, d=0.75):
    return max(m.big[v, w] - d, 0) / m.c[v] + lam(m, v, d) * p_cont(m, w)

METHODS = {"MLE": mle, "add-1": addk, "add-0.1": lambda m, w, v: addk(m, w, v, 0.1), "KN d=0.75": kn}

def check_sums(m):
    worst = 0.0
    for v in m.hist:
        for f in METHODS.values():
            worst = max(worst, abs(sum(f(m, w, v) for w in m.pred) - 1))
    return worst

worst = check_sums(M)
print(f"barcha kontekst va usullar boʻyicha yigʻindining 1 dan eng katta chetlanishi: {worst:.1e}")
assert worst < 1e-12, worst
print("✓ barcha taqsimotlar yigʻindisi 1 ga teng")
print()

c_v = M.c[CTX]
print("«{v}» konteksti (c = {c})".format(v=CTX, c=c_v), f"λ = {lam(M, CTX):.3f}")
rows = []
for w in sorted(M.pred, key=lambda w: (-M.big[CTX, w], -p_cont(M, w))):
    rows.append([w, M.big[CTX, w]] + [f(M, w, CTX) for f in METHODS.values()])
probs = pd.DataFrame(rows, columns=["w", "c"] + list(METHODS))
print(probs.head(6).round(3).to_string(index=False))
unseen = probs[probs["c"] == 0]
print("koʻrilmaganlar massasi:", ", ".join(f"{name} {unseen[name].sum():.2f}" for name in METHODS))

top = probs.head(6)
x = np.arange(len(top))
plt.figure(figsize=(8, 3))
for i, name in enumerate(METHODS):
    plt.bar(x + (i - 1.5) * 0.2, top[name], width=0.2, label=name)
plt.xticks(x, top["w"])
plt.title("P(w | " + CTX + ")")
plt.legend()
plt.tight_layout()
plt.show()

### Nima uchun chastota emas, davom ettirish

«Ulugʻbek» soʻzi 4 marta, «samarqandda» 3 marta uchraydi. Lekin «ulugʻbek» faqat «mirzo»dan keyin keladi, «samarqandda» esa uch xil soʻzdan keyin. Yangi kontekstda (ikkalasi ham uchramagan joyda) soʻz chastotasiga oddiy tayanish «ulugʻbek»ni, Kneser–Ney esa «samarqandda»ni afzal koʻradi. Ikkala ehtimol bir xil $\lambda(v)$ ga koʻpaytiriladi, shuning uchun kontekst ularning kattaligini oʻzgartiradi, tartibini emas. «Absolyut chegirma» (xuddi shu formula, lekin $P_{cont}$ oʻrniga soʻz chastotasi $P(w) = c(w)/N$) va Kneser–Neyni solishtiramiz.

In [ ]:
A, B = PAIR
rows = []
for w in (A, B):
    preds = sorted({v for (v, x) in M.big if x == w})
    rows.append([w, M.uni[w], M.n1_before[w], M.uni[w] / M.N, p_cont(M, w), ", ".join(preds)])
print(pd.DataFrame(rows, columns=["soʻz", "c(w)", "N1+(• w)", "P(w)", "P_cont(w)", "v"]).round(3).to_string(index=False))
print()

def abs_disc(m, w, v, d=0.75):
    # xuddi shu formula, lekin soʻz chastotasiga tayanadi
    return max(m.big[v, w] - d, 0) / m.c[v] + lam(m, v, d) * m.uni[w] / m.N

for v in NEW_CTXS:
    assert M.big[v, A] == 0 and M.big[v, B] == 0
    print(f"kontekst «{v}» (λ = {lam(M, v):.3f}):  chastotaga tayanish P({A})={abs_disc(M, A, v):.4f}  P({B})={abs_disc(M, B, v):.4f}   |   "
          f"Kneser–Ney P({A})={kn(M, A, v):.4f}  P({B})={kn(M, B, v):.4f}")

## 3. Perpleksiya

N ta bashorat qilingan tokendan (`</s>` bilan) iborat matndagi perpleksiya:
$$PP = 2^{-\frac{1}{N}\sum_i \log_2 P(w_i \mid w_{i-1})}.$$

Laboratoriyadagi test gaplarini olamiz. Agar bitta bigramma korpusda uchramagan boʻlsa, $P_{MLE} = 0$, $\log_2 0 = -\infty$ va MLE perpleksiyasi cheksiz.

In [ ]:
tests = ["onam samarqandda yashagan", "alisher navoiy samarqandda yashagan", "men choy ichaman", "onam choy ichaman"]

In [ ]:
def sentence_report(m, sentence, methods, show=True):
    toks = [BOS] + sentence.split() + [EOS]
    logs = {name: 0.0 for name in methods}
    rows = []
    for v, w in zip(toks, toks[1:]):
        row = [f"{v} → {w}", m.big[v, w]]
        for name, f in methods.items():
            p = f(m, w, v)
            logs[name] += math.log2(p) if p > 0 else -math.inf
            row.append(p)
        rows.append(row)
    n = len(toks) - 1
    pp = {name: 2 ** (-lp / n) for name, lp in logs.items()}
    if show:
        print(f"«{sentence}»  (N = {n})")
        print(pd.DataFrame(rows, columns=["bigramma", "c"] + list(methods)).round(3).to_string(index=False))
        print("   perpleksiya:", ", ".join(f"{k} = {v:.2f}" for k, v in pp.items()))
        print()
    return pp

lab_methods = {"MLE": mle, "add-1": addk, "KN d=0.75": kn}
for s in tests:
    sentence_report(M, s, lab_methods)

### Kattaroq korpus: k va d ni tanlash

9 ta gapda biror narsani baholash qiyin. Ularga xuddi shu mavzudagi (choy, shaharlar, oila, allomalar) yana 24 ta gap qoʻshamiz, 6 ta yangi gapni esa tekshirish uchun ajratib qoʻyamiz. Test gaplaridagi barcha soʻzlar lugʻatda bor, lekin baʼzi bigrammalar yoʻq. k va d ni ajratilgan gaplardagi perpleksiya boʻyicha tanlaymiz. Oʻz matnlaringizni «Files» paneli orqali yuklashingiz mumkin (masalan, `corpus.txt`, har qatorda bitta gap).

In [ ]:
more = [
    "otam choy ichadi", "otam qahva ichadi", "buvim koʻk choy ichadi", "bobom qora choy ichadi", "men koʻk choy ichaman",
    "alisher navoiy hirotda yashagan", "alisher navoiy buyuk mutafakkir edi", "mirzo ulugʻbek buyuk munajjim edi",
    "samarqand qadimiy shahar", "buxoro qadimiy shahar", "toshkent katta shahar", "samarqand katta shahar",
    "men toshkentda yashayman", "onam toshkentda yashaydi", "buvim samarqandda yashaydi", "bobom buxoroda yashaydi",
    "otam buxoroda oʻqigan", "men toshkentda oʻqiyman", "ertalab men choy ichaman", "kechqurun otam choy ichadi",
    "ertalab onam qahva ichadi", "buvim samarqandni yaxshi koʻradi", "men samarqandni yaxshi koʻraman",
    "otam buxoroni yaxshi koʻradi",
]
heldout = ["otam koʻk choy ichadi", "buvim toshkentda yashaydi", "ertalab otam qahva ichadi",
           "alisher navoiy buyuk olim edi", "bobom samarqandda yashaydi", "men buxoroni yaxshi koʻraman"]

In [ ]:
def corpus_pp(m, sentences, f):
    logp, n = 0.0, 0
    for s in sentences:
        toks = [BOS] + s.split() + [EOS]
        for v, w in zip(toks, toks[1:]):
            p = f(m, w, v)
            if p == 0:
                return math.inf
            logp += math.log2(p)
            n += 1
    return 2 ** (-logp / n)

M2 = BigramCounts(corpus + more)
print(f"{M2.n_sents} ta gap, {M2.N} ta token (<s> siz), |V| (soʻzlar + </s>) = {M2.V}, {M2.n1_all} ta turli bigramma")
assert check_sums(M2) < 1e-12
assert all(w in M2.pred for s in heldout for w in s.split())
for s in heldout:
    toks = [BOS] + s.split() + [EOS]
    missing = [f"{v} {w}" for v, w in zip(toks, toks[1:]) if M2.big[v, w] == 0]
    print(f"   {s:<32} koʻrilmagan bigrammalar: {missing}")
print()
print("MLE:", corpus_pp(M2, heldout, mle))

ks = [0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0]
ds = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.75, 0.9, 1.0]
pp_k = [corpus_pp(M2, heldout, lambda m, w, v: addk(m, w, v, k)) for k in ks]
pp_d = [corpus_pp(M2, heldout, lambda m, w, v: kn(m, w, v, d)) for d in ds]
best_k, best_d = ks[int(np.argmin(pp_k))], ds[int(np.argmin(pp_d))]
print(f"add-k: k=1 → {pp_k[ks.index(1.0)]:.2f};  eng yaxshi k = {best_k} → {min(pp_k):.2f}")
print(f"KN:    d=0.75 → {pp_d[ds.index(0.75)]:.2f};  eng yaxshi d = {best_d} → {min(pp_d):.2f}")

fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(ks, pp_k, "o-"); ax[0].set_xscale("log"); ax[0].set_xlabel("k"); ax[0].set_title("Ajratilgan gaplarda add-k perpleksiyasi")
ax[1].plot(ds, pp_d, "s-", color="#17907E"); ax[1].set_xlabel("d"); ax[1].set_title("Kneser–Ney perpleksiyasi")
for a in ax:
    a.set_ylabel("perpleksiya")
plt.tight_layout()
plt.show()

## 4. Harorat bilan matn yaratish

Model matn yaratadi: `<s>` dan boshlaymiz, $P(w \mid v)$ taqsimotini olamiz, tasodifiy soʻz tanlaymiz va `</s>` tushguncha takrorlaymiz. **Harorat** $T$ taqsimotni oʻzgartiradi: $p_i^{1/T}$, soʻng normallashtirish. $T < 1$ da taqsimot keskinroq (model koʻp uchraydiganini takrorlaydi), $T > 1$ da tekisroq (kutilmagan juftliklar koʻproq). Korpusda boʻlmagan bigrammalarni `*` bilan belgilaymiz: MLE ularni hech qachon yaratmaydi, Kneser–Ney esa ularga sezilarli ehtimol beradi, shuning uchun T = 1 da ham baʼzi gaplar gʻalati chiqadi. Hatto `<s> → </s>` juftligi (boʻsh gap) ham Kneser–Neyda taxminan 4 % ehtimol oladi: `</s>` koʻp turli soʻzlardan keyin keladi va uning $P_{cont}$ qiymati katta. «men choy ichadi» kabi gaplarga ham eʼtibor bering: har bir juftlik korpusda bor, lekin bigramma modeli «men» va «ichadi» orasidagi shaxs moslashuvini koʻrmaydi, chunki ular orasida bitta soʻz bor.

In [ ]:
rng = np.random.default_rng(24)

def next_dist(m, v, f, T=1.0):
    p = np.array([f(m, w, v) for w in m.pred])
    p = p ** (1 / T)
    return p / p.sum()

def generate(m, f, T=1.0, max_len=12, verbose=False):
    v, out = BOS, []
    while len(out) < max_len:
        p = next_dist(m, v, f, T)
        i = rng.choice(len(m.pred), p=p)
        w = m.pred[i]
        if verbose:
            top = np.argsort(-p)[:4]
            print(f"   {v:>10} → " + ", ".join(f"{m.pred[j]} {p[j]:.2f}" for j in top) + f"   ⇒ {w}")
        if w == EOS:
            break
        out.append(w + ("" if m.big[v, w] else "*"))
        v = w
    return " ".join(out) if out else "(darhol </s> tushdi: boʻsh gap)"

print("KN, T = 1:")
print("   ⇒", generate(M2, kn, 1.0, verbose=True))
print()
for T in (0.3, 1.0, 2.0):
    print(f"KN, harorat T = {T}")
    for _ in range(4):
        print("   ", generate(M2, kn, T))
print("MLE, harorat T = 1.0")
for _ in range(4):
    print("   ", generate(M2, mle, 1.0))
print("* — bigramma korpusda yoʻq")

v0 = GEN_CTX
fig, ax = plt.subplots(1, 3, figsize=(11, 3), sharey=True)
for a, T in zip(ax, (0.3, 1.0, 2.0)):
    p = next_dist(M2, v0, kn, T)
    top = np.argsort(-p)[:8]
    a.bar([M2.pred[j] for j in top], p[top], color="#2B55C9")
    a.set_title("T = " + str(T))
    a.tick_params(axis="x", rotation=60)
ax[0].set_ylabel("ehtimol")
fig.suptitle("«{v}»dan keyingi taqsimot".format(v=v0))
plt.tight_layout()
plt.show()

## 5. Xuddi shu `nltk.lm` da

`nltk.lm` da `MLE`, `Laplace`, `Lidstone` va `KneserNeyInterpolated` bor. Bizning koddan ikki farq: `nltk` lugʻatga `<s>` va `<UNK>` ni ham qoʻshadi, shuning uchun Laplasda maxraj $c(v) + |V| + 2$ (bizning $|V|$ ta soʻz boʻyicha yigʻindi 1 dan biroz kichik), Kneser–Neyda esa standart chegirma 0,1 — biz 0,75 qoʻyamiz. Ehtimollar va perpleksiyani biznikilar bilan solishtiramiz. Keyin oʻzimiz yozmagan trigramma Kneser–Ney modelini oʻqitamiz.

In [ ]:
from nltk.lm import MLE, Laplace, KneserNeyInterpolated
from nltk.lm.preprocessing import padded_everygram_pipeline

train_tokens = [s.split() for s in corpus + more]

def fit(model, order=2):
    data, vocab = padded_everygram_pipeline(order, train_tokens)
    model.fit(data, vocab)
    return model

lm_mle, lm_lap, lm_kn = fit(MLE(2)), fit(Laplace(2)), fit(KneserNeyInterpolated(2, discount=0.75))
print("nltk |vocab| =", len(lm_lap.vocab), "| bizning kod |V| =", M2.V)

held_bigrams = [(v, w) for s in heldout for v, w in zip([BOS] + s.split(), s.split() + [EOS])]
diff_kn = max(abs(lm_kn.score(w, [v]) - kn(M2, w, v)) for v, w in held_bigrams)
lap_nltk_formula = lambda m, w, v: (m.big[v, w] + 1) / (m.c[v] + m.V + 2)
diff_lap = max(abs(lm_lap.score(w, [v]) - lap_nltk_formula(M2, w, v)) for v, w in held_bigrams)
print(f"KN: ehtimollar orasidagi eng katta farq = {diff_kn:.1e}")
print(f"Laplace (c+1)/(c(v)+|V|+2): ehtimollar orasidagi eng katta farq = {diff_lap:.1e}")
v0 = M2.words[0]
print(f"nltk dagi P_Laplace ning bizning lugʻat boʻyicha yigʻindisi («{v0}»): {sum(lm_lap.score(w, [v0]) for w in M2.pred):.3f}")
print()
rows = [
    ("MLE", corpus_pp(M2, heldout, mle), lm_mle.perplexity(held_bigrams)),
    ("Laplace", corpus_pp(M2, heldout, addk), lm_lap.perplexity(held_bigrams)),
    ("KN d=0.75", corpus_pp(M2, heldout, kn), lm_kn.perplexity(held_bigrams)),
]
print(pd.DataFrame(rows, columns=["usul", "bizning kod", "nltk"]).round(3).to_string(index=False))

lm3 = fit(KneserNeyInterpolated(3, discount=0.75), order=3)
tri = [tuple(t[i - 2:i + 1]) for s in heldout for t in [[BOS, BOS] + s.split() + [EOS]] for i in range(2, len(t))]
print()
print(f"trigramma Kneser–Ney (nltk), perpleksiya: {lm3.perplexity(tri):.2f}")
gen = lm_kn.generate(12, text_seed=[BOS], random_seed=3)
print("nltk generatsiyasi:", " ".join(gen[:gen.index(EOS)] if EOS in gen else gen))   # nltk </s> da toʻxtamaydi: oʻzimiz kesamiz

## 6. Topshiriqlar

1. 1-boʻlimdagi korpusga «onam samarqandda yashaydi» gapini qoʻshing va «ulugʻbek» hamda «onam» kontekstlari uchun ehtimollar jadvalini qayta hisoblang. Yangi gapda «ulugʻbek» soʻzi boʻlmasa ham, undan keyingi taqsimot nima uchun oʻzgardi? (Maslahat: $P_{cont}$.)
2. Katta korpusdan $\lambda(v)$ (d = 0,75 da Kneser–Neyning tayanish uchun qoldiradigan massasi) eng katta boʻlgan kontekstni toping. Buni $N_{1+}(v\,\bullet)$ va $c(v)$ orqali tushuntiring.
3. **Ahmoqona tayanish** (stupid backoff) modelini yozing: $c(v\,w) > 0$ boʻlsa $S(w \mid v) = P_{MLE}(w \mid v)$, aks holda $0{,}4 \cdot P(w)$. Lugʻat boʻyicha yigʻindi 1 ga teng **emas**ligiga ishonch hosil qiling. Nima uchun bunday modelni perpleksiya bilan solishtirib boʻlmaydi?
4. Kneser–Ney modeli bilan T = 0,5 va T = 1,5 da 200 tadan gap yarating. Gapning oʻrtacha uzunligini va korpusda boʻlmagan bigrammalar ulushini solishtiring.
5. `nltk` da 2-, 3- va 4-tartibli Kneser–Ney modellarini katta korpusda oʻqiting va ularning `heldout` dagi perpleksiyasini solishtiring (n-tartib uchun n-grammalarni 5-boʻlimdagidek n − 1 ta `<s>` bilan tuzing). Bunday kichik korpusda uzun kontekst yordam beradimi? Nima uchun?

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing